## MOVIE  RECOMMENDATION SYSTEM 

Use of each dataset:
1. movies.csv - Metadata for Content-Based Filtering. 
2. rating.csv - Core data for Collaborative Filtering. 
3. tags.csv - User Tags.
4. links.csv - Links to fetch posters/synopses via the TMDB API later.

In [2]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
movies = pd.read_csv(r'C:\Abhishree\Projects_GitHubLinked\Movie Recommender\data\ml-latest-small\movies.csv')
ratings = pd.read_csv(r'C:\Abhishree\Projects_GitHubLinked\Movie Recommender\data\ml-latest-small\ratings.csv')
tags = pd.read_csv(r'C:\Abhishree\Projects_GitHubLinked\Movie Recommender\data\ml-latest-small\tags.csv')

In [4]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [6]:
tags.isnull().sum()

userId       0
movieId      0
tag          0
timestamp    0
dtype: int64

In [7]:
movies.isna().sum()

movieId    0
title      0
genres     0
dtype: int64

In [8]:
ratings.isna().sum()

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

### FEATURE ENGINEERING

>Get the clean title from the movie dataset.

In [9]:
movies['clean_title'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)

In [10]:
movies.head()

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II


>Clean the 'genres' column and remove this character -> '|'.

In [11]:
movies['clean_genre'] = movies['genres'].str.replace("|"," ", regex=False)
movies.head()

,movieId,title,genres,clean_title,clean_genre
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Comedy


> Group the tags of similar movie and name it tags_combined. 

In [12]:
tags_combined = tags.groupby('movieId')['tag'].apply(lambda x: " ".join(x)).reset_index()

In [13]:
tags_combined.head(10)

,movieId,tag
0,1,pixar pixar fun
1,2,fantasy magic board game Robin Williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake
5,11,politics president
6,14,politics president
7,16,Mafia
8,17,Jane Austen
9,21,Hollywood


In [14]:
tags.head()

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


> now merge this tags_combined column on the movies dataset. based on movieId.

> we will not hamper the original dataset and make the changes on the new dataset ( we create a new dataset )

In [15]:
movies_tags_combined = pd.merge(movies,tags_combined, on='movieId',how='left')

In [16]:
movies_tags_combined.head()

,movieId,title,genres,clean_title,clean_genre,tag
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,Adventure Animation Children Comedy Fantasy,pixar pixar fun
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,Adventure Children Fantasy,fantasy magic board game Robin Williams game
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,Comedy Romance,moldy old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,Comedy Drama Romance,NaN
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Comedy,pregnancy remake


In [17]:
movies_tags_combined.isna().sum()

movieId           0
title             0
genres            0
clean_title       0
clean_genre       0
tag            8170
dtype: int64

In [18]:
movies_tags_combined['tag'] = movies_tags_combined['tag'].fillna("")

In [19]:
movies_tags_combined.isna().sum()

movieId        0
title          0
genres         0
clean_title    0
clean_genre    0
tag            0
dtype: int64

> create a column which is the combination of genre and tags. 

In [20]:
movies_tags_combined['metadata'] = movies_tags_combined['clean_genre']+" " +movies_tags_combined['tag']

In [21]:
movies_tags_combined.head()

,movieId,title,genres,clean_title,clean_genre,tag,metadata
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,Adventure Animation Children Comedy Fantasy,pixar pixar fun,Adventure Animation Children Comedy Fantasy pi...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,Adventure Children Fantasy,fantasy magic board game Robin Williams game,Adventure Children Fantasy fantasy magic board...
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,Comedy Romance,moldy old,Comedy Romance moldy old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,Comedy Drama Romance,,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Comedy,pregnancy remake,Comedy pregnancy remake


> Now we focus on ratings.

In [22]:
ratings.head(2)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247


>Group ratings together and find their avg. 

>also the count of rating. as rating_count =2 is less reliable than rating_count=200.

In [23]:
ratings_stats = ratings.groupby('movieId').agg(
    avg_rating = ('rating','mean'),
    rating_count = ('rating','count')
).reset_index()

In [24]:
ratings_stats.head(8)

,movieId,avg_rating,rating_count
0,1,3.920930,215
1,2,3.431818,110
2,3,3.259615,52
3,4,2.357143,7
4,5,3.071429,49
5,6,3.946078,102
6,7,3.185185,54
7,8,2.875000,8


>Merge this column with the movie_tags_combined dataset.

In [25]:
movies_tags_combined = pd.merge(movies_tags_combined,ratings_stats,on='movieId',how='left')

In [26]:
movies_tags_combined.head()

,movieId,title,genres,clean_title,clean_genre,tag,metadata,avg_rating,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,Adventure Animation Children Comedy Fantasy,pixar pixar fun,Adventure Animation Children Comedy Fantasy pi...,3.920930,215.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,Adventure Children Fantasy,fantasy magic board game Robin Williams game,Adventure Children Fantasy fantasy magic board...,3.431818,110.0
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,Comedy Romance,moldy old,Comedy Romance moldy old,3.259615,52.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,Comedy Drama Romance,,Comedy Drama Romance,2.357143,7.0
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Comedy,pregnancy remake,Comedy pregnancy remake,3.071429,49.0


> There must exist some nan values so fill those.

In [27]:
movies_tags_combined.isna().sum()

movieId          0
title            0
genres           0
clean_title      0
clean_genre      0
tag              0
metadata         0
avg_rating      18
rating_count    18
dtype: int64

In [28]:
movies_tags_combined['avg_rating'] = movies_tags_combined['avg_rating'].fillna(0)

In [29]:
movies_tags_combined['rating_count'] = movies_tags_combined['rating_count'].fillna(0) 

In [30]:
movies_tags_combined.isna().sum()

movieId         0
title           0
genres          0
clean_title     0
clean_genre     0
tag             0
metadata        0
avg_rating      0
rating_count    0
dtype: int64

In [31]:
movies_tags_combined.head()

,movieId,title,genres,clean_title,clean_genre,tag,metadata,avg_rating,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,Adventure Animation Children Comedy Fantasy,pixar pixar fun,Adventure Animation Children Comedy Fantasy pi...,3.920930,215.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,Adventure Children Fantasy,fantasy magic board game Robin Williams game,Adventure Children Fantasy fantasy magic board...,3.431818,110.0
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,Comedy Romance,moldy old,Comedy Romance moldy old,3.259615,52.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,Comedy Drama Romance,,Comedy Drama Romance,2.357143,7.0
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,Comedy,pregnancy remake,Comedy pregnancy remake,3.071429,49.0


> Dropping unnecessary features such as genres.

In [32]:
movies_tags_combined.drop(columns='genres',inplace=True)

In [33]:
movies_tags_combined.head()

,movieId,title,clean_title,clean_genre,tag,metadata,avg_rating,rating_count
0,1,Toy Story (1995),Toy Story,Adventure Animation Children Comedy Fantasy,pixar pixar fun,Adventure Animation Children Comedy Fantasy pi...,3.920930,215.0
1,2,Jumanji (1995),Jumanji,Adventure Children Fantasy,fantasy magic board game Robin Williams game,Adventure Children Fantasy fantasy magic board...,3.431818,110.0
2,3,Grumpier Old Men (1995),Grumpier Old Men,Comedy Romance,moldy old,Comedy Romance moldy old,3.259615,52.0
3,4,Waiting to Exhale (1995),Waiting to Exhale,Comedy Drama Romance,,Comedy Drama Romance,2.357143,7.0
4,5,Father of the Bride Part II (1995),Father of the Bride Part II,Comedy,pregnancy remake,Comedy pregnancy remake,3.071429,49.0


> Normalize avg_rating and rating_counts.

In [34]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

movies_tags_combined[['norm_avg_rating','norm_rating_count']] = scaler.fit_transform(movies_tags_combined[['avg_rating','rating_count']])

In [35]:
movies_tags_combined.head()

,movieId,title,clean_title,clean_genre,tag,metadata,avg_rating,rating_count,norm_avg_rating,norm_rating_count
0,1,Toy Story (1995),Toy Story,Adventure Animation Children Comedy Fantasy,pixar pixar fun,Adventure Animation Children Comedy Fantasy pi...,3.920930,215.0,0.784186,0.653495
1,2,Jumanji (1995),Jumanji,Adventure Children Fantasy,fantasy magic board game Robin Williams game,Adventure Children Fantasy fantasy magic board...,3.431818,110.0,0.686364,0.334347
2,3,Grumpier Old Men (1995),Grumpier Old Men,Comedy Romance,moldy old,Comedy Romance moldy old,3.259615,52.0,0.651923,0.158055
3,4,Waiting to Exhale (1995),Waiting to Exhale,Comedy Drama Romance,,Comedy Drama Romance,2.357143,7.0,0.471429,0.021277
4,5,Father of the Bride Part II (1995),Father of the Bride Part II,Comedy,pregnancy remake,Comedy pregnancy remake,3.071429,49.0,0.614286,0.148936


## CONTENT BASED RECOMMENDATION

1. convert words/features to numbers(vectors) and then compare the angle between them.
STEP A: TF-IDF
* It converts text into numerical vectors based on how important a word is to a document relative to all documents.
* TF-IDF(t, d, D) = TF(t, d)*IDF(t, D)
* TF (Term Frequency): How often a word appears in a single item description.
* IDF (Inverse Document Frequency): Penalizes common words (like "the", "movie", "a") that appear across every item.

### VECTORIZATION
Turn text into numbers. We use TF-IDF

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer 

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies_tags_combined['metadata'])
tfidf_matrix.shape

(9742, 1748)

### COSINE SIMILARITY
* Calculate the similarity between all movies. 
* If two movies have identical genres, their similarity score is 1.0. 
* If they share no genres at all, their score is 0.0

In [37]:
from sklearn.metrics.pairwise import linear_kernel

cosine_sim = linear_kernel(tfidf_matrix,tfidf_matrix)
cosine_sim.shape

(9742, 9742)

### RECOMMENDATION SYSTEM

In [38]:
movies_tags_combined.head()

,movieId,title,clean_title,clean_genre,tag,metadata,avg_rating,rating_count,norm_avg_rating,norm_rating_count
0,1,Toy Story (1995),Toy Story,Adventure Animation Children Comedy Fantasy,pixar pixar fun,Adventure Animation Children Comedy Fantasy pi...,3.920930,215.0,0.784186,0.653495
1,2,Jumanji (1995),Jumanji,Adventure Children Fantasy,fantasy magic board game Robin Williams game,Adventure Children Fantasy fantasy magic board...,3.431818,110.0,0.686364,0.334347
2,3,Grumpier Old Men (1995),Grumpier Old Men,Comedy Romance,moldy old,Comedy Romance moldy old,3.259615,52.0,0.651923,0.158055
3,4,Waiting to Exhale (1995),Waiting to Exhale,Comedy Drama Romance,,Comedy Drama Romance,2.357143,7.0,0.471429,0.021277
4,5,Father of the Bride Part II (1995),Father of the Bride Part II,Comedy,pregnancy remake,Comedy pregnancy remake,3.071429,49.0,0.614286,0.148936


In [39]:
def recomender(title,top_n =5): 
    matches = movies_tags_combined[movies_tags_combined['clean_title'].str.contains(title,case=False,regex=False)]
    if matches.empty:
        return 'Movie not found'
    movie_idx = matches.index[0]

    movie_score = list(enumerate(cosine_sim[movie_idx]))

    sorted_score = sorted(movie_score, key= lambda x: x[1], reverse=True)
    
    top_movie = [i[0] for i in sorted_score[1:top_n+1]] 

    return movies_tags_combined['clean_title'].iloc[top_movie]

In [40]:
print(recomender("charlie's angels"))

328     Naked Gun 33 1/3: The Final Insult
344                Low Down Dirty Shame, A
1102                   Beverly Hills Ninja
1206                           Money Talks
1361         Mr. Nice Guy (Yat goh ho yan)
Name: clean_title, dtype: str
